# flight-llm-debrief: fine-tune on Colab (T4)
Runtime → Change runtime type → **T4 GPU**. Generates the simulated fleet, builds the SFT dataset, LoRA-fine-tunes Qwen2.5-1.5B-Instruct, and evaluates against the baselines.

In [ ]:
!git clone https://github.com//flight-llm-debrief.git
%cd flight-llm-debrief
!pip -q install -e ".[train]"

In [ ]:
# 1) data: more flights = better model (2000 flights ≈ 4k examples, ~20 min on Colab CPU)
!flightlake simulate --n 1000 --out data/raw --workers 2
!flightlake ingest data/raw --lake data/lake --workers 2
!flightdebrief build-dataset --lake data/lake --out data/sft

In [ ]:
# 2) baselines (what the LLM must beat)
!flightdebrief baselines --data data/sft --out results

In [ ]:
# 3) LoRA fine-tune
!python -m flightdebrief.train_sft --config configs/qwen2.5-1.5b-lora.yaml

In [ ]:
# 4) evaluate base vs fine-tuned
!flightdebrief eval-model --backend hf --model Qwen/Qwen2.5-1.5B-Instruct --name base_zero_shot --limit 60
!flightdebrief eval-model --backend hf --model Qwen/Qwen2.5-1.5B-Instruct --adapter outputs/qwen2.5-1.5b-debrief/final_adapter --name finetuned

In [ ]:
import json, glob
for f in sorted(glob.glob('results/eval_*.json')):
    r = json.load(open(f)); print(r['name'], {k: r['silent'][k] for k in ('fault_acc','macro_f1','onset_mae_s','grounded_rate','json_valid')})

In [ ]:
# 5) optional: push adapter to the Hub
# from huggingface_hub import login; login()
# from peft import PeftModel; ...  model.push_to_hub('YOUR_HF_USER/qwen2.5-1.5b-flight-debrief')